In [ ]:
# import pandas as pd

# # 檔案名稱列表
# file_names = [
#     "summary_results_no_index.csv",
#     "summary_results_second_half.csv",
#     "summary_results_second_half_last.csv",
#     "summary_results_second_half_last_2.csv",
# ]

# # 讀取並合併符合條件的數據
# filtered_dfs = []
# for file in file_names:
#     df = pd.read_csv(file)
#     filtered_df = df[df["Top K"].isin([5, 7, 10])]
#     filtered_dfs.append(filtered_df)

# # 合併所有篩選後的數據
# final_df = pd.concat(filtered_dfs, ignore_index=True)

# # 排序數據
# final_df = final_df.sort_values(
#     by=["Chunk Size", "Chunk Overlap", "Top K"], ascending=True
# )

# # 計算新的欄位 (Chunk Size + Chunk Overlap) * Top K
# final_df["Tokens"] = (
#     final_df["Chunk Size"] + final_df["Chunk Overlap"]
# ) * final_df["Top K"]

# # 儲存到 CSV
# final_df.to_csv("final_eval_results.csv", index=False)

# # 輸出總行數
# print(f"Total rows in final_eval_results.csv: {len(final_df)}")

Total rows in final_eval_results.csv: 108


In [ ]:
import pandas as pd

import pandas as pd


def display_summary_table_weighted(
    relevancy_weight=0.4, correctness_weight=0.4, response_time_weight=0.2
):
    """
    使用加權總分計算最佳參數組合，允許用戶自定義權重：
    Final Score = (Relevancy Weight * Relevancy) + (Correctness Weight * Correctness) - (Response Time Weight * Response Time)

    :param relevancy_weight: 相關性權重 (預設 0.4)
    :param correctness_weight: 正確性權重 (預設 0.4)
    :param response_time_weight: 反應時間權重 (預設 0.2，因為時間越短越好，所以會減去這個值)
    """

    print("\n--- Summary of All Parameter Combinations (Weighted Score) ---")
    print(
        f"Weights: Relevancy = {relevancy_weight}, Correctness = {correctness_weight}, Response Time = {response_time_weight}"
    )

    # 讀取 final_eval_results.csv
    summary_results_df = pd.read_csv("final_eval_results.csv")

    # 計算加權總分
    summary_results_df["Final Score"] = (
        relevancy_weight * summary_results_df["Relevancy Score"]
        + correctness_weight * summary_results_df["Correctness Score"]
        - response_time_weight * summary_results_df["Avg Response Time"]
    )

    # 按 Final Score 降序排序
    sorted_df = summary_results_df.sort_values(by=["Final Score"], ascending=False)

    # 設定表格完整顯示
    pd.set_option("display.max_columns", None)
    pd.set_option("display.expand_frame_repr", False)
    pd.set_option("display.max_rows", None)
    pd.reset_option("display.float_format")

    # 讓索引從 1 開始
    sorted_df.index = range(1, len(sorted_df) + 1)

    # 顯示 Top 5 最佳組合
    print("\nTop 5 Best Parameter Combinations (Weighted Score):")
    print(sorted_df.head(5).to_string())

    # # 顯示完整表格
    # print(sorted_df.to_string())

    # 找出最佳參數組合
    best_row = sorted_df.iloc[0]
    print("\nBest Parameter Combination (Weighted Score):")
    print(f"Chunk Size: {best_row['Chunk Size']}")
    print(f"Chunk Overlap: {best_row['Chunk Overlap']}")
    print(f"Top K: {best_row['Top K']}")
    print(f"Relevancy Score: {best_row['Relevancy Score']}")
    print(f"Correctness Score: {best_row['Correctness Score']}")
    print(f"Avg Response Time: {best_row['Avg Response Time']} seconds")
    print(f"Final Score: {best_row['Final Score']}")


def display_summary_table_ranked():
    """
    使用排名加總法評估最佳參數組合：
    每個參數分數個別排名，最後相加，總排名越低代表整體表現越好
    """
    print("\n--- Summary of All Parameter Combinations (Ranked Score) ---")

    # 讀取 final_eval_results.csv
    summary_results_df = pd.read_csv("final_eval_results.csv")

    # 分別排名（越小越好）
    summary_results_df["Relevancy Rank"] = summary_results_df["Relevancy Score"].rank(
        ascending=False, method="min"
    )
    summary_results_df["Correctness Rank"] = summary_results_df["Correctness Score"].rank(ascending=False)
    summary_results_df["Response Time Rank"] = summary_results_df["Avg Response Time"].rank(ascending=True)  # 時間越短越好

    # 計算總排名分數
    summary_results_df["Total Rank Score"] = (
        summary_results_df["Relevancy Rank"] +
        summary_results_df["Correctness Rank"] +
        summary_results_df["Response Time Rank"]
    )

    # 按總排名分數排序（分數越小代表整體表現越好）
    sorted_df = summary_results_df.sort_values(by=["Total Rank Score"], ascending=True)

    # 設定表格完整顯示
    pd.set_option("display.max_columns", None)
    pd.set_option("display.expand_frame_repr", False)
    pd.set_option("display.max_rows", None)
    pd.reset_option("display.float_format")

    # 讓索引從 1 開始
    sorted_df.index = range(1, len(sorted_df) + 1)

    # 顯示 Top 5 最佳組合
    print("\nTop 5 Best Parameter Combinations (Ranked Score):")
    print(sorted_df.head(5).to_string())

    # # 顯示完整表格
    # print(sorted_df.to_string())

    # 找出最佳參數組合
    best_row = sorted_df.iloc[0]
    print("\nBest Parameter Combination (Ranked Score):")
    print(f"Chunk Size: {best_row['Chunk Size']}")
    print(f"Chunk Overlap: {best_row['Chunk Overlap']}")
    print(f"Top K: {best_row['Top K']}")
    print(f"Relevancy Score: {best_row['Relevancy Score']}")
    print(f"Correctness Score: {best_row['Correctness Score']}")
    print(f"Avg Response Time: {best_row['Avg Response Time']} seconds")
    print(f"Total Rank Score: {best_row['Total Rank Score']}")

# 呼叫
display_summary_table_weighted()
display_summary_table_weighted(relevancy_weight=0.45, correctness_weight=0.45, response_time_weight=0.1)
display_summary_table_ranked()


--- Summary of All Parameter Combinations (Weighted Score) ---
Weights: Relevancy = 0.4, Correctness = 0.4, Response Time = 0.2

Top 5 Best Parameter Combinations (Weighted Score):
   Chunk Size  Chunk Overlap  Top K  Relevancy Score  Correctness Score  Avg Response Time  Tokens  Final Score
1         192             28      7           0.9667             3.9000             0.8567    1540      1.77534
2         192             19      7           0.9667             3.9000             0.9052    1477      1.76564
3         192             38      7           1.0000             3.8667             0.9166    1610      1.76336
4         384             57      7           0.9667             4.0333             1.2819    3087      1.74362
5         128             19      7           0.9000             3.8667             0.8566    1029      1.73536

Best Parameter Combination (Weighted Score):
Chunk Size: 192.0
Chunk Overlap: 28.0
Top K: 7.0
Relevancy Score: 0.9667
Correctness Score: 3.9
Avg 

In [4]:
import pandas as pd

import pandas as pd


def display_summary_table_weighted(
    relevancy_weight=0.45,
    correctness_weight=0.45,
    response_time_weight=0.05,
    tokens_weight=0.05,
):
    """
    使用加權總分計算最佳參數組合，允許用戶自定義權重：
    Final Score = (Relevancy Weight * Relevancy) + (Correctness Weight * Correctness)
                  - (Response Time Weight * Response Time) - (Tokens Weight * Tokens)
    :param relevancy_weight: 相關性權重 (預設 0.4)
    :param correctness_weight: 正確性權重 (預設 0.4)
    :param response_time_weight: 反應時間權重 (預設 0.2，因為時間越短越好，所以會減去這個值)
    """

    print("\n--- Summary of All Parameter Combinations (Weighted Score) ---")
    print(
        f"Weights: Relevancy = {relevancy_weight}, Correctness = {correctness_weight}, Response Time = {response_time_weight}, Tokens = {tokens_weight}"
    )

    # 設定表格完整顯示
    pd.set_option("display.max_columns", None)
    pd.set_option("display.expand_frame_repr", False)
    pd.set_option("display.max_rows", None)
    pd.reset_option("display.float_format")

    # 讀取 final_eval_results.csv
    summary_results_df = pd.read_csv("final_eval_results.csv")

    # 計算加權總分
    summary_results_df["Final Score"] = (
        relevancy_weight * summary_results_df["Relevancy Score"]
        + correctness_weight * summary_results_df["Correctness Score"]
        - response_time_weight * summary_results_df["Avg Response Time"]
        - tokens_weight * summary_results_df["Tokens"]
    )

    # 按 Final Score 降序排序
    sorted_df = summary_results_df.sort_values(by=["Final Score"], ascending=False)

    # 讓索引從 1 開始
    sorted_df.index = range(1, len(sorted_df) + 1)

    # 顯示 Top 5 最佳組合
    print("\nTop 5 Best Parameter Combinations (Weighted Score):")
    print(sorted_df.head(5).to_string())

    # # 顯示完整表格
    # print(sorted_df.to_string())

    # 找出最佳參數組合
    best_row = sorted_df.iloc[0]
    print("\nBest Parameter Combination (Weighted Score):")
    print(f"Chunk Size: {best_row['Chunk Size']}")
    print(f"Chunk Overlap: {best_row['Chunk Overlap']}")
    print(f"Top K: {best_row['Top K']}")
    print(f"Relevancy Score: {best_row['Relevancy Score']}")
    print(f"Correctness Score: {best_row['Correctness Score']}")
    print(f"Avg Response Time: {best_row['Avg Response Time']} seconds")
    print(f"Tokens: {best_row['Tokens']}")
    print(f"Final Score: {best_row['Final Score']}")


def display_summary_table_ranked():
    """
    使用排名加總法評估最佳參數組合：
    每個參數分數個別排名，最後相加，總排名越低代表整體表現越好
    """
    print("\n--- Summary of All Parameter Combinations (Ranked Score) ---")

    # 設定表格完整顯示
    pd.set_option("display.max_columns", None)
    pd.set_option("display.expand_frame_repr", False)
    pd.set_option("display.max_rows", None)
    pd.reset_option("display.float_format")

    # 讀取 final_eval_results.csv
    summary_results_df = pd.read_csv("final_eval_results.csv")

    # 分別排名（越小越好）
    summary_results_df["Relevancy Rank"] = summary_results_df["Relevancy Score"].rank(
        ascending=False, method="min"
    )
    summary_results_df["Correctness Rank"] = summary_results_df[
        "Correctness Score"
    ].rank(ascending=False)
    summary_results_df["Response Time Rank"] = summary_results_df[
        "Avg Response Time"
    ].rank(
        ascending=True
    )  # 時間越短越好
    summary_results_df["Tokens Rank"] = summary_results_df["Tokens"].rank(
        ascending=True
    )  # Tokens 越少越好

    # 計算總排名分數
    summary_results_df["Total Rank Score"] = (
        summary_results_df["Relevancy Rank"]
        + summary_results_df["Correctness Rank"]
        + summary_results_df["Response Time Rank"]
        + summary_results_df["Tokens Rank"]
    )

    # 按總排名分數排序（分數越小代表整體表現越好）
    sorted_df = summary_results_df.sort_values(by=["Total Rank Score"], ascending=True)

    # 讓索引從 1 開始
    sorted_df.index = range(1, len(sorted_df) + 1)

    # 顯示 Top 5 最佳組合
    print("\nTop 5 Best Parameter Combinations (Ranked Score):")
    print(sorted_df.head(5).to_string())

    # # 顯示完整表格
    # print(sorted_df.to_string())

    # 找出最佳參數組合
    best_row = sorted_df.iloc[0]
    print("\nBest Parameter Combination (Ranked Score):")
    print(f"Chunk Size: {best_row['Chunk Size']}")
    print(f"Chunk Overlap: {best_row['Chunk Overlap']}")
    print(f"Top K: {best_row['Top K']}")
    print(f"Relevancy Score: {best_row['Relevancy Score']}")
    print(f"Correctness Score: {best_row['Correctness Score']}")
    print(f"Avg Response Time: {best_row['Avg Response Time']} seconds")
    print(f"Tokens: {best_row['Tokens']}")
    print(f"Total Rank Score: {best_row['Total Rank Score']}")


# 呼叫
display_summary_table_weighted()
display_summary_table_weighted(
    relevancy_weight=0.4,
    correctness_weight=0.4,
    response_time_weight=0.1,
    tokens_weight=0.1,
)
display_summary_table_ranked()


--- Summary of All Parameter Combinations (Weighted Score) ---
Weights: Relevancy = 0.45, Correctness = 0.45, Response Time = 0.05, Tokens = 0.05

Top 5 Best Parameter Combinations (Weighted Score):
   Chunk Size  Chunk Overlap  Top K  Relevancy Score  Correctness Score  Avg Response Time  Tokens  Final Score
1         128              6      5           0.9333             3.8000             1.3226     670   -31.436145
2         128             12      5           0.9000             3.7667             1.2203     700   -32.961000
3         128             19      5           0.9333             3.7667             0.7471     735   -34.672355
4         128             25      5           1.0000             3.6333             0.7882     765   -36.204425
5         128              6      7           0.9333             3.8000             1.3326     938   -44.836645

Best Parameter Combination (Weighted Score):
Chunk Size: 128.0
Chunk Overlap: 6.0
Top K: 5.0
Relevancy Score: 0.9333
Correctnes

In [ ]:
import pandas as pd

import pandas as pd


def display_summary_table_weighted(
    relevancy_weight=0.4, correctness_weight=0.4, response_time_weight=0.2
):
    """
    使用加權總分計算最佳參數組合，允許用戶自定義權重：
    Final Score = (Relevancy Weight * Relevancy) + (Correctness Weight * Correctness) - (Response Time Weight * Response Time)

    :param relevancy_weight: 相關性權重 (預設 0.4)
    :param correctness_weight: 正確性權重 (預設 0.4)
    :param response_time_weight: 反應時間權重 (預設 0.2，因為時間越短越好，所以會減去這個值)
    """

    print("\n--- Summary of All Parameter Combinations (Weighted Score) ---")
    print(
        f"Weights: Relevancy = {relevancy_weight}, Correctness = {correctness_weight}, Response Time = {response_time_weight}"
    )

    # 讀取 final_eval_results.csv
    summary_results_df = pd.read_csv("final_eval_results.csv")

    # 計算加權總分
    summary_results_df["Final Score"] = (
        relevancy_weight * summary_results_df["Relevancy Score"]
        + correctness_weight * summary_results_df["Correctness Score"]
        - response_time_weight * summary_results_df["Avg Response Time"]
    )

    # 按 Final Score 降序排序
    sorted_df = summary_results_df.sort_values(by=["Final Score"], ascending=False)

    # 設定表格完整顯示
    pd.set_option("display.max_columns", None)
    pd.set_option("display.expand_frame_repr", False)
    pd.set_option("display.max_rows", None)
    pd.reset_option("display.float_format")

    # 讓索引從 1 開始
    sorted_df.index = range(1, len(sorted_df) + 1)

    # 取得 Top 5 最佳組合
    top_5_df = sorted_df.head(5)

    # 顯示 Top 5 最佳組合
    print("\nTop 5 Best Parameter Combinations (Weighted Score):")
    print(top_5_df.to_string())

    # 再次依照 Tokens 進行排序（升序或降序）
    top_5_df = top_5_df.sort_values(by=["Tokens"], ascending=True)  # 升序，可改為 False 降序

    # 顯示 Top 5 最佳組合
    print("\nTop 5 Best Parameter Combinations (Weighted Score, then Tokens):")
    print(top_5_df.to_string())

    # # 顯示完整表格
    # print(sorted_df.to_string())

    # 找出最佳參數組合
    best_row = sorted_df.iloc[0]
    print("\nBest Parameter Combination (Weighted Score):")
    print(f"Chunk Size: {best_row['Chunk Size']}")
    print(f"Chunk Overlap: {best_row['Chunk Overlap']}")
    print(f"Top K: {best_row['Top K']}")
    print(f"Relevancy Score: {best_row['Relevancy Score']}")
    print(f"Correctness Score: {best_row['Correctness Score']}")
    print(f"Avg Response Time: {best_row['Avg Response Time']} seconds")
    print(f"Tokens: {best_row['Tokens']}")
    print(f"Final Score: {best_row['Final Score']}")

    best_row = top_5_df.iloc[0]
    print("\nBest Parameter Combination (Weighted Score, then Tokens):")
    print(f"Chunk Size: {best_row['Chunk Size']}")
    print(f"Chunk Overlap: {best_row['Chunk Overlap']}")
    print(f"Top K: {best_row['Top K']}")
    print(f"Relevancy Score: {best_row['Relevancy Score']}")
    print(f"Correctness Score: {best_row['Correctness Score']}")
    print(f"Avg Response Time: {best_row['Avg Response Time']} seconds")
    print(f"Tokens: {best_row['Tokens']}")
    print(f"Final Score: {best_row['Final Score']}")

    print('---------------------------------')
def display_summary_table_ranked():
    """
    使用排名加總法評估最佳參數組合：
    每個參數分數個別排名，最後相加，總排名越低代表整體表現越好
    """
    print("\n--- Summary of All Parameter Combinations (Ranked Score) ---")

    # 讀取 final_eval_results.csv
    summary_results_df = pd.read_csv("final_eval_results.csv")

    # 分別排名（越小越好）
    summary_results_df["Relevancy Rank"] = summary_results_df["Relevancy Score"].rank(
        ascending=False, method="min"
    )
    summary_results_df["Correctness Rank"] = summary_results_df[
        "Correctness Score"
    ].rank(ascending=False)
    summary_results_df["Response Time Rank"] = summary_results_df[
        "Avg Response Time"
    ].rank(
        ascending=True
    )  # 時間越短越好

    # 計算總排名分數
    summary_results_df["Total Rank Score"] = (
        summary_results_df["Relevancy Rank"]
        + summary_results_df["Correctness Rank"]
        + summary_results_df["Response Time Rank"]
    )

    # 按總排名分數排序（分數越小代表整體表現越好）
    sorted_df = summary_results_df.sort_values(by=["Total Rank Score"], ascending=True)

    # 設定表格完整顯示
    pd.set_option("display.max_columns", None)
    pd.set_option("display.expand_frame_repr", False)
    pd.set_option("display.max_rows", None)
    pd.reset_option("display.float_format")

    # 讓索引從 1 開始
    sorted_df.index = range(1, len(sorted_df) + 1)

    # 取得 Top 5 最佳組合
    top_5_df = sorted_df.head(10)
    # 顯示 Top 5 最佳組合

    # 顯示 Top 5 最佳組合
    print("\nTop 5 Best Parameter Combinations (Ranked Score):")
    print(top_5_df.to_string())

    # 再次依照 Tokens 進行排序（升序或降序）
    top_5_df = top_5_df.sort_values(
        by=["Tokens"], ascending=True
    )  # 升序，可改為 False 降序

    # 顯示 Top 5 最佳組合
    print("\nTop 5 Best Parameter Combinations (Ranked Score, then Tokens):")
    print(top_5_df.to_string())

    # # 顯示完整表格
    # print(sorted_df.to_string())

    # 找出最佳參數組合
    best_row = sorted_df.iloc[0]
    print("\nBest Parameter Combination (Ranked Score):")
    print(f"Chunk Size: {best_row['Chunk Size']}")
    print(f"Chunk Overlap: {best_row['Chunk Overlap']}")
    print(f"Top K: {best_row['Top K']}")
    print(f"Relevancy Score: {best_row['Relevancy Score']}")
    print(f"Correctness Score: {best_row['Correctness Score']}")
    print(f"Avg Response Time: {best_row['Avg Response Time']} seconds")
    print(f"Tokens: {best_row['Tokens']}")
    print(f"Total Rank Score: {best_row['Total Rank Score']}")

    best_row = top_5_df.iloc[0]
    print("\nBest Parameter Combination (Ranked Score, then Tokens):")
    print(f"Chunk Size: {best_row['Chunk Size']}")
    print(f"Chunk Overlap: {best_row['Chunk Overlap']}")
    print(f"Top K: {best_row['Top K']}")
    print(f"Relevancy Score: {best_row['Relevancy Score']}")
    print(f"Correctness Score: {best_row['Correctness Score']}")
    print(f"Avg Response Time: {best_row['Avg Response Time']} seconds")
    print(f"Tokens: {best_row['Tokens']}")
    print(f"Total Rank Score: {best_row['Total Rank Score']}")
    print("---------------------------------")


# 呼叫
display_summary_table_weighted()
display_summary_table_weighted(
    relevancy_weight=0.45, correctness_weight=0.45, response_time_weight=0.1
)
display_summary_table_ranked()


--- Summary of All Parameter Combinations (Weighted Score) ---
Weights: Relevancy = 0.4, Correctness = 0.4, Response Time = 0.2

Top 5 Best Parameter Combinations (Weighted Score):
     Chunk Size  Chunk Overlap  Top K  Relevancy Score  Correctness Score  Avg Response Time  Tokens  Final Score
1     192         28             7     0.9667           3.9000              0.8567             1540   1.77534    
2     192         19             7     0.9667           3.9000              0.9052             1477   1.76564    
3     192         38             7     1.0000           3.8667              0.9166             1610   1.76336    
4     384         57             7     0.9667           4.0333              1.2819             3087   1.74362    
5     128         19             7     0.9000           3.8667              0.8566             1029   1.73536    
6     256         25             7     0.9333           3.9333              1.0757             1967   1.73150    
7     128         19

In [9]:
import pandas as pd

import pandas as pd


def display_summary_table_weighted(
    relevancy_weight=0.4, correctness_weight=0.4, response_time_weight=0.2
):
    """
    使用加權總分計算最佳參數組合，允許用戶自定義權重：
    Final Score = (Relevancy Weight * Relevancy) + (Correctness Weight * Correctness) - (Response Time Weight * Response Time)

    :param relevancy_weight: 相關性權重 (預設 0.4)
    :param correctness_weight: 正確性權重 (預設 0.4)
    :param response_time_weight: 反應時間權重 (預設 0.2，因為時間越短越好，所以會減去這個值)
    """

    print("\n--- Summary of All Parameter Combinations (Weighted Score) ---")
    print(
        f"Weights: Relevancy = {relevancy_weight}, Correctness = {correctness_weight}, Response Time = {response_time_weight}"
    )

    # 讀取 final_eval_results.csv
    summary_results_df = pd.read_csv("final_eval_results.csv")

    # 計算加權總分
    summary_results_df["Final Score"] = (
        relevancy_weight * summary_results_df["Relevancy Score"]
        + correctness_weight * summary_results_df["Correctness Score"]
        - response_time_weight * summary_results_df["Avg Response Time"]
    )

    # 按 Final Score 降序排序
    sorted_df = summary_results_df.sort_values(by=["Final Score"], ascending=False)

    # 設定表格完整顯示
    pd.set_option("display.max_columns", None)
    pd.set_option("display.expand_frame_repr", False)
    pd.set_option("display.max_rows", None)
    pd.reset_option("display.float_format")

    # 讓索引從 1 開始
    sorted_df.index = range(1, len(sorted_df) + 1)

    # 取得 Top 5 最佳組合
    top_5_df = sorted_df.head(5)

    # 顯示 Top 5 最佳組合
    print("\nTop 5 Best Parameter Combinations (Weighted Score):")
    print(top_5_df.to_string())

    # 再次依照 Tokens 進行排序（升序或降序）
    top_5_df = top_5_df.sort_values(
        by=["Tokens"], ascending=True
    )  # 升序，可改為 False 降序

    # 顯示 Top 5 最佳組合
    print("\nTop 5 Best Parameter Combinations (Weighted Score, then Tokens):")
    print(top_5_df.to_string())

    # # 顯示完整表格
    # print(sorted_df.to_string())

    # 找出最佳參數組合
    best_row = sorted_df.iloc[0]
    print("\nBest Parameter Combination (Weighted Score):")
    print(f"Chunk Size: {best_row['Chunk Size']}")
    print(f"Chunk Overlap: {best_row['Chunk Overlap']}")
    print(f"Top K: {best_row['Top K']}")
    print(f"Relevancy Score: {best_row['Relevancy Score']}")
    print(f"Correctness Score: {best_row['Correctness Score']}")
    print(f"Avg Response Time: {best_row['Avg Response Time']} seconds")
    print(f"Tokens: {best_row['Tokens']}")
    print(f"Final Score: {best_row['Final Score']}")

    best_row = top_5_df.iloc[0]
    print("\nBest Parameter Combination (Weighted Score, then Tokens):")
    print(f"Chunk Size: {best_row['Chunk Size']}")
    print(f"Chunk Overlap: {best_row['Chunk Overlap']}")
    print(f"Top K: {best_row['Top K']}")
    print(f"Relevancy Score: {best_row['Relevancy Score']}")
    print(f"Correctness Score: {best_row['Correctness Score']}")
    print(f"Avg Response Time: {best_row['Avg Response Time']} seconds")
    print(f"Tokens: {best_row['Tokens']}")
    print(f"Final Score: {best_row['Final Score']}")

    print("---------------------------------")


def display_summary_table_ranked():
    """
    使用排名加總法評估最佳參數組合：
    每個參數分數個別排名，最後相加，總排名越低代表整體表現越好
    """
    print("\n--- Summary of All Parameter Combinations (Ranked Score) ---")

    # 讀取 final_eval_results.csv
    summary_results_df = pd.read_csv("final_eval_results.csv")

    # 分別排名（越小越好）
    summary_results_df["Relevancy Rank"] = summary_results_df["Relevancy Score"].rank(
        ascending=False, method="min"
    )
    summary_results_df["Correctness Rank"] = summary_results_df[
        "Correctness Score"
    ].rank(ascending=False)
    summary_results_df["Response Time Rank"] = summary_results_df[
        "Avg Response Time"
    ].rank(
        ascending=True
    )  # 時間越短越好

    # 計算總排名分數
    summary_results_df["Total Rank Score"] = (
        summary_results_df["Relevancy Rank"]
        + summary_results_df["Correctness Rank"]
        + summary_results_df["Response Time Rank"]
    )

    # 按總排名分數排序（分數越小代表整體表現越好）
    sorted_df = summary_results_df.sort_values(by=["Total Rank Score"], ascending=True)

    # 設定表格完整顯示
    pd.set_option("display.max_columns", None)
    pd.set_option("display.expand_frame_repr", False)
    pd.set_option("display.max_rows", None)
    pd.reset_option("display.float_format")

    # 讓索引從 1 開始
    sorted_df.index = range(1, len(sorted_df) + 1)

    # 取得 Top 5 最佳組合
    top_5_df = sorted_df.head(10)
    # 顯示 Top 5 最佳組合

    # 顯示 Top 5 最佳組合
    print("\nTop 5 Best Parameter Combinations (Ranked Score):")
    print(top_5_df.to_string())

    # 再次依照 Tokens 進行排序（升序或降序）
    top_5_df = top_5_df.sort_values(
        by=["Tokens"], ascending=True
    )  # 升序，可改為 False 降序

    # 顯示 Top 5 最佳組合
    print("\nTop 5 Best Parameter Combinations (Ranked Score, then Tokens):")
    print(top_5_df.to_string())

    # # 顯示完整表格
    # print(sorted_df.to_string())

    # 找出最佳參數組合
    best_row = sorted_df.iloc[0]
    print("\nBest Parameter Combination (Ranked Score):")
    print(f"Chunk Size: {best_row['Chunk Size']}")
    print(f"Chunk Overlap: {best_row['Chunk Overlap']}")
    print(f"Top K: {best_row['Top K']}")
    print(f"Relevancy Score: {best_row['Relevancy Score']}")
    print(f"Correctness Score: {best_row['Correctness Score']}")
    print(f"Avg Response Time: {best_row['Avg Response Time']} seconds")
    print(f"Tokens: {best_row['Tokens']}")
    print(f"Total Rank Score: {best_row['Total Rank Score']}")

    best_row = top_5_df.iloc[0]
    print("\nBest Parameter Combination (Ranked Score, then Tokens):")
    print(f"Chunk Size: {best_row['Chunk Size']}")
    print(f"Chunk Overlap: {best_row['Chunk Overlap']}")
    print(f"Top K: {best_row['Top K']}")
    print(f"Relevancy Score: {best_row['Relevancy Score']}")
    print(f"Correctness Score: {best_row['Correctness Score']}")
    print(f"Avg Response Time: {best_row['Avg Response Time']} seconds")
    print(f"Tokens: {best_row['Tokens']}")
    print(f"Total Rank Score: {best_row['Total Rank Score']}")
    print("---------------------------------")


# 呼叫
display_summary_table_weighted()
display_summary_table_weighted(
    relevancy_weight=0.45, correctness_weight=0.45, response_time_weight=0.1
)
display_summary_table_ranked()


--- Summary of All Parameter Combinations (Weighted Score) ---
Weights: Relevancy = 0.4, Correctness = 0.4, Response Time = 0.2

Top 5 Best Parameter Combinations (Weighted Score):
   Chunk Size  Chunk Overlap  Top K  Relevancy Score  Correctness Score  Avg Response Time  Tokens  Final Score
1  192         28             7      0.9667           3.9000             0.8567             1540    1.77534    
2  192         19             7      0.9667           3.9000             0.9052             1477    1.76564    
3  192         38             7      1.0000           3.8667             0.9166             1610    1.76336    
4  384         57             7      0.9667           4.0333             1.2819             3087    1.74362    
5  128         19             7      0.9000           3.8667             0.8566             1029    1.73536    

Top 5 Best Parameter Combinations (Weighted Score, then Tokens):
   Chunk Size  Chunk Overlap  Top K  Relevancy Score  Correctness Score  Avg Res

In [ ]:
import pandas as pd

# 載入CSV檔案
df = pd.read_csv("final_eval_results.csv")

# 原始數據
print(f"原始數據: 共 {len(df)} 筆資料。")

# 計算Relevancy Score的75百分位數（前25%的閾值）
relevancy_threshold = df["Relevancy Score"].quantile(0.75)
# 計算Correctness Score的75百分位數（前25%的閾值）
correctness_threshold = df["Correctness Score"].quantile(0.75)

# 印出門檻值，以便了解篩選標準
print(f"Relevancy Score 門檻值（前25%）: {relevancy_threshold}")
print(f"Correctness Score 門檻值（前25%）: {correctness_threshold}")

# 篩選出同時滿足兩個得分都在前25%的資料列
filtered_df = df[
    (df["Relevancy Score"] >= relevancy_threshold)
    & (df["Correctness Score"] >= correctness_threshold)
]
# 儲存篩選後的數據為新的 CSV 檔案
filtered_df.to_csv("filtered_top_25_scores.csv", index=False)



原始數據: 共 108 筆資料。
Relevancy Score 門檻值（前25%）: 0.9667
Correctness Score 門檻值（前25%）: 3.8667
成功篩選數據。共找到 11 筆符合條件的資料。


In [ ]:
import pandas as pd

# 載入CSV檔案
df = pd.read_csv("final_eval_results.csv")

# 原始數據
print(f"原始數據: 共 {len(df)} 筆資料。")

# 計算Relevancy Score的75百分位數（前25%的閾值）
relevancy_threshold = df["Relevancy Score"].quantile(0.75)
# 計算Correctness Score的75百分位數（前25%的閾值）
correctness_threshold = df["Correctness Score"].quantile(0.75)

# 印出門檻值，以便了解篩選標準
print(f"Relevancy Score 門檻值（前25%）: {relevancy_threshold}")
print(f"Correctness Score 門檻值（前25%）: {correctness_threshold}")

# 篩選出同時滿足兩個得分都在前25%的資料列
filtered_df = df[
    (df["Relevancy Score"] >= relevancy_threshold)
    & (df["Correctness Score"] >= correctness_threshold)
]

# 儲存篩選後的數據為新的 CSV 檔案
filtered_df.to_csv("filtered_top_25_scores.csv", index=False)

# 印出處理結果摘要
print(f"成功篩選數據。共找到 {len(filtered_df)} 筆符合條件的資料。")
print("---------------------------------")

# 依照 Avg Response Time 排序並印出表格
print("\n按照 Avg Response Time 排序的結果:")
time_sorted_df = filtered_df.sort_values("Avg Response Time")
print(time_sorted_df.to_string(index=False))
best_time_row = time_sorted_df.iloc[0]
print("\n最佳響應時間參數組合:")
print(f"Chunk Size: {best_time_row['Chunk Size']}")
print(f"Chunk Overlap: {best_time_row['Chunk Overlap']}")
print(f"Top K: {best_time_row['Top K']}")
print(f"Relevancy Score: {best_time_row['Relevancy Score']}")
print(f"Correctness Score: {best_time_row['Correctness Score']}")
print(f"Avg Response Time: {best_time_row['Avg Response Time']} seconds")
print(f"Tokens: {best_time_row['Tokens']}")

print('---------------------------------')
# 依照 Tokens 排序並印出表格
print("\n按照 Tokens 排序的結果:")
token_sorted_df = filtered_df.sort_values("Tokens")
print(token_sorted_df.to_string(index=False))
best_token_row = token_sorted_df.iloc[0]
print("\n最佳Token用量參數組合:")
print(f"Chunk Size: {best_token_row['Chunk Size']}")
print(f"Chunk Overlap: {best_token_row['Chunk Overlap']}")
print(f"Top K: {best_token_row['Top K']}")
print(f"Relevancy Score: {best_token_row['Relevancy Score']}")
print(f"Correctness Score: {best_token_row['Correctness Score']}")
print(f"Avg Response Time: {best_token_row['Avg Response Time']} seconds")
print(f"Tokens: {best_token_row['Tokens']}")

原始數據: 共 108 筆資料。
Relevancy Score 門檻值（前25%）: 0.9333
Correctness Score 門檻值（前25%）: 3.6667
成功篩選數據。共找到 37 筆符合條件的資料。
---------------------------------

按照 Avg Response Time 排序的結果:
 Chunk Size  Chunk Overlap  Top K  Relevancy Score  Correctness Score  Avg Response Time  Tokens
        128             19      5           0.9333             3.7667             0.7471     735
        128             25      7           0.9667             3.7667             0.8429    1071
        192             28      7           0.9667             3.9000             0.8567    1540
        128             12     10           0.9333             3.7000             0.8879    1400
        192             19      7           0.9667             3.9000             0.9052    1477
        192             38      7           1.0000             3.8667             0.9166    1610
        128             25     10           0.9667             3.8000             0.9207    1530
        384             57      5           0.9667